In [1]:
import pandas as pd
import numpy as np
import os
import time
from PIL import Image
from typing import List, Dict
from collections import Counter

In [2]:
# Loads images from the rsif-2022-0588-File007/dataset into a dictionary list.
data: List[List] = []

dataset_path = "rsif-2022-0588-File007/dataset"

for label in os.listdir(dataset_path):
    label_path = os.path.join(dataset_path, label)
    
    for file in os.listdir(label_path):
        img_path = os.path.join(label_path, file)
        img = Image.open(img_path)
        data.append([label, img])

# Creates a panda dataframe and converts the images to a numpy numerical array, resized to 32x32 and normalized / 255.0
df = pd.DataFrame(data, columns=["label", "image"])
df["image"] = df["image"].apply(
    lambda img: np.array(img.convert("RGB").resize((32,32)), dtype=np.float32) / 255.0
)

# k-Nearest Neighbors (k-NN)

In [3]:
class kNN:
    def __init__(self, df=pd.DataFrame, test_frac=0.2, metric="euclidean"):
        # Store the dataframe, test fraction, and distance metric
        self.df = df
        self.test_frac = test_frac
        self.metric = metric

        # Initialize train/test data
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None

    # Euclidean distance: sqrt(sum((x1 - x2)^2))
    def _euclidean(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2) ** 2))

    # norm distance: sum(|x1 - x2|)
    def _norm(self, x1, x2):
        return np.sum(np.abs(x1 - x2))

    # Max distance: \max(|x1 - x2|)
    def _max(self, x1, x2):
        return np.max(np.abs(x1 - x2))

    # Discrete distance: 0 if equal, 1 if not
    def _discrete(self, x1, x2):
        return 0 if np.array_equal(x1, x2) else 1

    # train_test_split
    def train_test_split(self):
        # Shuffle the dataframe
        df_shuffled = self.df.sample(frac=1, random_state=1).reset_index(drop=True)
        n_test = int(len(df_shuffled) * self.test_frac)

        # Test set first n_test rows
        df_test = df_shuffled.iloc[:n_test]

        # Train set remaining rows
        df_train = df_shuffled.iloc[n_test:]

        # Flatten images into 1D vectors, maybe another type of reshape would be possible?
        self.X_train = np.stack(df_train["image"].values).reshape(len(df_train), -1)
        self.y_train = df_train["label"].values
        self.X_test = np.stack(df_test["image"].values).reshape(len(df_test), -1)
        self.y_test = df_test["label"].values

    # Predict a single sample
    def _predict_single(self, x, k):
        # Compute distances to all training points based on the chosen metric
        match self.metric:
            case "euclidean":
                distances = [self._euclidean(x, xi) for xi in self.X_train]
            case "norm":
                distances = [self._norm(x, xi) for xi in self.X_train]
            case "max":
                distances = [self._max(x, xi) for xi in self.X_train]
            case "discrete":
                distances = [self._discrete(x, xi) for xi in self.X_train]
            case _:
                # Raise an error if the metric is unsupported
                raise ValueError(f"Unsupported metric: {self.metric}")

        # Get indices of k nearest neighbors
        k_indices = np.argsort(distances)[:k]

        # Get labels of k nearest neighbors
        k_labels = [self.y_train[i] for i in k_indices]

        # Return the most common label from the neighbors
        most_common = Counter(k_labels).most_common(1)
        return most_common[0][0]

    # Predict from multiple samples
    def predict(self, X, k):
        """
            Predict labels for multiple samples
            X: (n_samples, n_features)
        """
        return np.array([self._predict_single(x, k) for x in X])

    # Run k-NN for k = 1 to 7 and report accuracy
    def run(self):
        # Store each runtime
        runtime = []
        # First, split the data
        self.train_test_split()

        # Loop over different values of k
        for k in range(1, 8):
            # Fit is trivial here since we just store training data
            start_time = time.time()
            predictions = self.predict(self.X_test, k)
            accuracy = np.mean(predictions == self.y_test)
            end_time = time.time()
            runtime.append(end_time - start_time)
            print(f"Test accuracy for k={k} with metric {self.metric}: {accuracy:.2f}")
        return runtime

In [4]:
knn_euclidean = kNN(test_frac=0.5, df=df, metric="euclidean")

In [5]:
euclidean_runtime = knn_euclidean.run()

Test accuracy for k=1 with metric euclidean: 0.76
Test accuracy for k=2 with metric euclidean: 0.76
Test accuracy for k=3 with metric euclidean: 0.76
Test accuracy for k=4 with metric euclidean: 0.77
Test accuracy for k=5 with metric euclidean: 0.78
Test accuracy for k=6 with metric euclidean: 0.77
Test accuracy for k=7 with metric euclidean: 0.78


In [6]:
knn_norm = kNN(test_frac=0.5, df=df, metric="norm")

In [7]:
norm_runtime = knn_norm.run()

Test accuracy for k=1 with metric norm: 0.76
Test accuracy for k=2 with metric norm: 0.76
Test accuracy for k=3 with metric norm: 0.77
Test accuracy for k=4 with metric norm: 0.77
Test accuracy for k=5 with metric norm: 0.77
Test accuracy for k=6 with metric norm: 0.76
Test accuracy for k=7 with metric norm: 0.78


In [8]:
knn_max = kNN(test_frac=0.5, df=df, metric="max")

In [9]:
max_runtime = knn_max.run()

Test accuracy for k=1 with metric max: 0.59
Test accuracy for k=2 with metric max: 0.59
Test accuracy for k=3 with metric max: 0.58
Test accuracy for k=4 with metric max: 0.58
Test accuracy for k=5 with metric max: 0.58
Test accuracy for k=6 with metric max: 0.59
Test accuracy for k=7 with metric max: 0.58


In [10]:
knn_discrete = kNN(test_frac=0.5, df=df, metric="discrete")

In [11]:
discrete_runtime = knn_discrete.run()

Test accuracy for k=1 with metric discrete: 0.58
Test accuracy for k=2 with metric discrete: 0.58
Test accuracy for k=3 with metric discrete: 0.58
Test accuracy for k=4 with metric discrete: 0.58
Test accuracy for k=5 with metric discrete: 0.58
Test accuracy for k=6 with metric discrete: 0.58
Test accuracy for k=7 with metric discrete: 0.58


In [12]:
print(f"Mean runtime euclidean metric k=1...7: {np.mean(euclidean_runtime)}")
print(f"Mean runtime norm metric k=1...7: {np.mean(norm_runtime)}")
print(f"Mean runtime max metric k=1...7: {np.mean(max_runtime)}")
print(f"Mean runtime discrete metric k=1...7: {np.mean(discrete_runtime)}")

Mean runtime euclidean metric k=1...7: 10.824871982846942
Mean runtime norm metric k=1...7: 9.049752099173409
Mean runtime max metric k=1...7: 8.32563727242606
Mean runtime discrete metric k=1...7: 4.793751955032349


# Results for k-NN

The results obtained with the k-NN algorithm indicate that the choice of distance metric has a significant impact on classification performance. In particular, the Euclidean and Norm distance metric shows better accuracy.

In contrast, the maximum distance and discrete metrics exhibit poorer performance, with accuracy levels close to random guessing (approximately 50%). This suggests that these metrics are less effective at capturing meaningful similarity relationships within the dataset.

In addition to classification accuracy, the runtime performance of the k-NN algorithm was also evaluated. Since k-NN doesn't seen to be a learning method, it does not involve an explicit training phase, most of the computational cost occurs during classification. The metric choice influences runtimem. Simpler metrics such as the discrete distance may be computationally cheaper but do not provide meaningful performance gains (~6 seconds) but do lower classification accuracy.

## Further Reading

Abdel-Basset, Mohamed, et al. *A Comprehensive Review of k-Nearest Neighbor Algorithm and Its Applications*. arXiv, 2017.  
https://arxiv.org/abs/1708.04321